# 05 · Protótipo do Produto — Hit Maker (Spotify)

Módulo anterior: [04 · Ideação da Solução](04_ideacao_solucao.ipynb).

Implementa a **Frente 4** das GQs refinadas (Aplicação de Produto e Inteligência de Mercado) — a camada que transforma o modelo/clusters do módulo 04 em algo que um artista, gravadora ou curador poderia efetivamente usar:

1. Comparar uma faixa específica ao "perfil de hit" do seu gênero.
2. Listar faixas de baixa popularidade com alta probabilidade prevista de hit (via modelo).
3. Listar faixas de baixa popularidade estruturalmente parecidas com hits do gênero (via distância, sem depender do modelo supervisionado).

Lê os artefatos gerados no módulo 04 (`df_clusterizado.csv`, `modelo_rf.pkl`, `scaler_cluster.pkl`).

In [ ]:
import pickle
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.neighbors import NearestNeighbors

warnings.filterwarnings("ignore")

COLUNAS = {
    "genero": "track_genre",
    "popularidade": "popularity",
    "id_faixa": "track_id",
    "nome_faixa": "track_name",
    "artista": "artists",
}

FEATURES_NUMERICAS = [
    "danceability", "energy", "valence", "tempo", "loudness",
    "acousticness", "instrumentalness", "liveness", "speechiness",
]

ARTIFACTS_DIR = Path("artifacts")
df_clusterizado = pd.read_csv(ARTIFACTS_DIR / "df_clusterizado.csv")
with open(ARTIFACTS_DIR / "modelo_rf.pkl", "rb") as f:
    modelo = pickle.load(f)
with open(ARTIFACTS_DIR / "scaler_cluster.pkl", "rb") as f:
    scaler_cluster = pickle.load(f)

df_clusterizado.shape

## Frente 4: Aplicação de Produto e Inteligência de Mercado

GQ: *É viável criar um perfil de referência por gênero que permita comparar uma música específica com os hits da categoria? O modelo consegue mapear faixas subestimadas (baixa popularidade atual, mas alta similaridade estrutural aos hits)?*

Este módulo é deliberadamente um **protótipo**, não um produto — a interface é a saída de funções Python, não uma UI. O objetivo é validar se a lógica de negócio funciona antes de investir em formato de entrega (dashboard, relatório, API — decisão em aberto, ver módulo 02, GQ de Produto).

In [ ]:
def criar_perfil_referencia(df: pd.DataFrame, genero: str) -> pd.Series:
    """
    GQ: 'É viável criar um perfil de referência por gênero que permita
    comparar uma música específica com os hits da categoria?'
    """
    hits_do_genero = df[(df[COLUNAS["genero"]] == genero) & (df["is_hit"] == 1)]
    # ATENÇÃO: gêneros com poucos hits (amostra pequena) geram perfis
    # instáveis. Defina um mínimo de faixas (ex: 30) antes de confiar no perfil.
    if len(hits_do_genero) < 30:
        warnings.warn(f"Gênero '{genero}' tem apenas {len(hits_do_genero)} hits "
                       f"— perfil pode não ser estatisticamente robusto.")
    return hits_do_genero[FEATURES_NUMERICAS].mean()

In [ ]:
def comparar_musica_com_perfil(musica: pd.Series, perfil_referencia: pd.Series) -> pd.DataFrame:
    """
    GQ: 'Identificar pontos de aproximação/diferença e recomendar melhorias.'
    Retorna, feature a feature, a diferença percentual entre a faixa e o hit médio.
    # ATENÇÃO: se alguma feature do perfil de referência for 0 (ex.: instrumentalness
    # médio nulo em certos gêneros), a divisão abaixo gera inf/NaN — trate esse caso
    # antes de usar o resultado em produção (ex.: substituir por diferença absoluta).
    """
    comparacao = pd.DataFrame({
        "musica": musica[FEATURES_NUMERICAS],
        "perfil_hit_genero": perfil_referencia,
    })
    comparacao["diferenca_%"] = (
        (comparacao["musica"] - comparacao["perfil_hit_genero"])
        / comparacao["perfil_hit_genero"] * 100
    )
    return comparacao.sort_values("diferenca_%", key=abs, ascending=False)

### Reconstruindo `X` para o modelo

O módulo 04 treinou o modelo sobre uma matriz `X` específica (features numéricas + one-hot de gênero). Para reutilizar o modelo aqui, refazemos essa mesma transformação — `preparar_features` é reaproveitada do pipeline original.

In [ ]:
def preparar_features(df: pd.DataFrame, incluir_genero: bool = True):
    """Monta a matriz de features (X) e o alvo (y) para o modelo."""
    X = df[FEATURES_NUMERICAS].copy()
    if incluir_genero:
        # ATENÇÃO: one-hot em gêneros com alta cardinalidade pode gerar
        # muitas colunas esparsas. Avalie target/frequency encoding se
        # o dataset tiver dezenas de gêneros.
        genero_dummies = pd.get_dummies(df[COLUNAS["genero"]], prefix="genero")
        X = pd.concat([X, genero_dummies], axis=1)
    y = df["is_hit"]
    return X, y


X, y = preparar_features(df_clusterizado)
X.shape

In [ ]:
def identificar_faixas_subestimadas(df: pd.DataFrame, modelo, X: pd.DataFrame,
                                     percentil_popularidade_baixa: float = 30,
                                     limiar_probabilidade_hit: float = 0.6):
    """
    GQ: 'O modelo consegue mapear faixas subestimadas (baixa popularidade
    atual, mas alta similaridade estrutural aos hits)?'

    Lógica: pega faixas com popularidade real baixa, mas que o modelo
    classifica com alta probabilidade de ser hit — candidatas a
    relançamento/investimento.
    """
    # ATENÇÃO: aqui há um risco de vazamento de dados (data leakage) se X
    # tiver sido usado no treino do mesmo `modelo` — sempre rode isso sobre
    # dados fora do conjunto de treino, ou refaça o predict_proba em cima do
    # dataset completo com um modelo já validado.
    probas = modelo.predict_proba(X)[:, 1]
    limiar_pop_baixa = np.percentile(df[COLUNAS["popularidade"]], percentil_popularidade_baixa)

    df_resultado = df.copy()
    df_resultado["probabilidade_hit"] = probas
    candidatas = df_resultado[
        (df_resultado[COLUNAS["popularidade"]] <= limiar_pop_baixa) &
        (df_resultado["probabilidade_hit"] >= limiar_probabilidade_hit)
    ].sort_values("probabilidade_hit", ascending=False)

    return candidatas[[COLUNAS["nome_faixa"], COLUNAS["artista"], COLUNAS["genero"],
                        COLUNAS["popularidade"], "probabilidade_hit"]]

In [ ]:
def musicas_similares_a_hits(df: pd.DataFrame, cluster_scaler, colunas: list = None,
                              n_vizinhos: int = 5, genero: str = None):
    """
    GQ: 'mapear faixas subestimadas com alta similaridade estrutural aos
    hits' — versão por DISTÂNCIA (KNN), complementar à versão por
    probabilidade do modelo supervisionado.

    Para cada faixa NÃO-hit, encontra os N hits mais próximos no espaço de
    features e calcula a distância média — quanto menor, mais "parecida
    com hit" a faixa é, mesmo tendo popularidade baixa hoje.
    """
    colunas = colunas or FEATURES_NUMERICAS
    base = df if genero is None else df[df[COLUNAS["genero"]] == genero]

    hits = base[base["is_hit"] == 1]
    nao_hits = base[base["is_hit"] == 0]

    # ATENÇÃO: reaproveita o MESMO scaler ajustado nos hits para manter a
    # mesma escala entre os dois grupos — nunca ajuste um scaler novo em
    # cada subconjunto, ou a distância deixa de ser comparável.
    # ATENÇÃO (corrigido): se `hits` estiver vazio (ex.: gênero sem hits
    # suficientes), NearestNeighbors falha — valide `len(hits) > 0` antes
    # de chamar esta função em produção.
    X_hits = cluster_scaler.transform(hits[colunas])
    X_nao_hits = cluster_scaler.transform(nao_hits[colunas])

    knn = NearestNeighbors(n_neighbors=min(n_vizinhos, len(hits)))
    knn.fit(X_hits)
    distancias, _ = knn.kneighbors(X_nao_hits)

    resultado = nao_hits.copy()
    resultado["distancia_media_a_hits"] = distancias.mean(axis=1)
    colunas_saida = [COLUNAS["nome_faixa"], COLUNAS["artista"], COLUNAS["genero"],
                      COLUNAS["popularidade"], "distancia_media_a_hits"]
    return resultado.sort_values("distancia_media_a_hits")[colunas_saida]

## Demonstração ponta a ponta (gênero: `pop`)

Escolha ilustrativa de gênero — troque `GENERO_DEMO` pelo gênero real que quiser investigar (mesmo `# ATENÇÃO` do pipeline original).

In [ ]:
# ATENÇÃO: troque "pop" pelo gênero real que quiser investigar.
GENERO_DEMO = "pop"

perfil_pop = criar_perfil_referencia(df_clusterizado, genero=GENERO_DEMO)
perfil_pop

In [ ]:
# Exemplo: compara a primeira faixa não-hit do gênero escolhido com o perfil de hits do gênero.
faixa_exemplo = df_clusterizado[
    (df_clusterizado[COLUNAS["genero"]] == GENERO_DEMO) & (df_clusterizado["is_hit"] == 0)
].iloc[0]

comparar_musica_com_perfil(faixa_exemplo, perfil_pop)

### Faixas subestimadas — via probabilidade do modelo

> ⚠️ **Limitação conhecida, não resolvida neste protótipo:** `identificar_faixas_subestimadas` roda aqui sobre o mesmo `X` usado no treino do `modelo` (módulo 04), o que caracteriza vazamento de dados (decisão em aberto nº 8 do módulo 02). Numa versão além de protótipo, isso precisa rodar sobre um conjunto de validação separado, nunca visto no treino.

In [ ]:
faixas_subestimadas = identificar_faixas_subestimadas(df_clusterizado, modelo, X)
faixas_subestimadas.head(20)

### Faixas subestimadas — via similaridade estrutural (KNN, sem modelo supervisionado)

Versão complementar: não depende do `RandomForestClassifier`, só da distância no espaço de features escalonadas — útil como segunda opinião, já que os dois métodos podem discordar.

In [ ]:
candidatas_por_similaridade = musicas_similares_a_hits(
    df_clusterizado, scaler_cluster, genero=GENERO_DEMO, n_vizinhos=5
)
candidatas_por_similaridade.head(20)

## Do protótipo ao produto: o que falta decidir

Estas funções respondem às GQs da Frente 4 no nível de lógica de negócio. Para virar produto, faltam decisões levantadas no módulo [02 · Requisitos](02_requisitos.ipynb) e na seção "Produto" de `docs/README.md`:

- **Formato de entrega:** dashboard interativo, relatório estático, ou API de score? (GQ `[NOVA]` de Produto, sem decisão registrada ainda.)
- **Persona/momento de uso:** artista avaliando uma faixa pré-lançamento (sem `popularity` real disponível) vs. gravadora auditando o catálogo já lançado — mudam quais destas 3 funções fazem sentido em cada caso.
- **Explicabilidade:** a GQ de Ética pede que o score venha acompanhado de explicação (feature importance / SHAP), não só um número — hoje `importancia_features` (módulo 04) existe, mas não está conectada à saída deste módulo por faixa individual.
- **KPI de produto:** distinto da métrica de modelo (ROC-AUC do módulo 04) — ainda não definido.
- **Correção do vazamento de dados** apontado acima, antes de qualquer uso além de demonstração interna.

## Status do fluxo completo

| Módulo | Status |
|---|---|
| [01 · Guiding Questions](01_guiding_questions.ipynb) | ✅ Consolidado |
| [02 · Requisitos](02_requisitos.ipynb) | ✅ Levantado — 10 decisões de negócio + 6 requisitos éticos em aberto |
| [03 · Tratamento e Limpeza dos Dados](03_tratamento_limpeza_dados.ipynb) | ✅ Implementado sobre `archive/dataset.csv` |
| [04 · Ideação da Solução](04_ideacao_solucao.ipynb) | ✅ Implementado — EDA, modelo preditivo, clustering |
| 05 · Protótipo do Produto | ✅ Implementado — nível protótipo, com limitações documentadas acima |
| Frente Ética | ❌ Não iniciada |